# Stage 2 — Damage Segmentation (YOLOv8s-seg, 4 classes)

**Project:** car-damage-morocco — Stage 2/3.

Trains `yolov8s-seg` on the Roboflow `is_it_damaged` v6 dataset, remapped from 7 to 4 classes: `dent`, `scratch`, `glass`, `broken_part`.

**Hardware:** Kaggle T4 (single GPU). Expected runtime: ~1.5–2h for 60 epochs.

**Prerequisite:** Kaggle Secret `ROBOFLOW_API_KEY` must be attached (Add-ons → Secrets).

**Outputs (in `/kaggle/working/`):**
- `damage_seg_best.pt` — the model you'll download and place at `models/stage2/best.pt`
- `runs/damage_seg_v1/` — training plots, confusion matrix, validation predictions

## 1. Clean install + GPU check
Single consolidated install at the top — no upgrade/downgrade cycles that break imports.

In [ ]:
import sys, subprocess

def pip(*args):
    return subprocess.run([sys.executable, '-m', 'pip', *args], capture_output=True, text=True)

pip('uninstall', '-y', 'ray')
pip('install', '-q', '--no-warn-conflicts', 'ultralytics==8.3.40', 'roboflow')

for mod in list(sys.modules):
    if (mod == 'ray' or mod.startswith('ray.')
        or mod == 'ultralytics' or mod.startswith('ultralytics.')):
        del sys.modules[mod]

import os, shutil, yaml
from pathlib import Path

import numpy as np
import torch

if not hasattr(np, 'trapz') and hasattr(np, 'trapezoid'):
    np.trapz = np.trapezoid
    print('Applied np.trapz -> np.trapezoid shim for numpy 2.x')

import ultralytics
from ultralytics import YOLO

print(f'NumPy:       {np.__version__}')
print(f'Ultralytics: {ultralytics.__version__}')
print(f'PyTorch:     {torch.__version__}')
print(f'CUDA:        {torch.cuda.is_available()}')
assert torch.cuda.is_available(), 'GPU required'
print(f'GPU:         {torch.cuda.get_device_name(0)}')


## 2. Download the Roboflow `is_it_damaged` v6 dataset

In [ ]:
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

RF_API_KEY = UserSecretsClient().get_secret('ROBOFLOW_API_KEY')

RF_WORKSPACE = 'martin-rpfil'
RF_PROJECT   = 'is_it_damaged'
RF_VERSION   = 6

rf = Roboflow(api_key=RF_API_KEY)
project = rf.workspace(RF_WORKSPACE).project(RF_PROJECT)
dataset = project.version(RF_VERSION).download('yolov8', location='/kaggle/working/damage_raw')

DAMAGE_RAW = Path(dataset.location)
print(f"Downloaded to: {DAMAGE_RAW}")
print('\nOriginal data.yaml:')
print((DAMAGE_RAW / 'data.yaml').read_text())

## 3. Remap 7 classes → 4 classes

| Original | Our class | Rationale |
|---|---|---|
| dent          | dent        | direct |
| scratch       | scratch     | direct |
| glass shatter | glass       | windshield/window only |
| crack         | broken_part | structural damage |
| lamp broken   | broken_part | headlight replacement |
| tire flat     | broken_part | tire replacement |
| background    | (skipped)   | not a damage |

In [ ]:
with open(DAMAGE_RAW / 'data.yaml') as f:
    orig_yaml = yaml.safe_load(f)

orig_names = orig_yaml['names']
if isinstance(orig_names, dict):
    orig_names = [orig_names[i] for i in sorted(orig_names.keys())]

print(f"Original {len(orig_names)} classes:")
for i, n in enumerate(orig_names):
    print(f"  {i}: {n}")

NAME_TO_NEW = {
    'dent':          0,
    'scratch':       1,
    'glass shatter': 2,
    'crack':         3,
    'lamp broken':   3,
    'tire flat':     3,
    'background':    None,
}

REMAP = {}
SKIP_CLASSES = set()
for old_idx, old_name in enumerate(orig_names):
    key = old_name.lower().strip()
    if key not in NAME_TO_NEW:
        raise ValueError(f"Unmapped class: '{old_name}' (idx {old_idx})")
    new_idx = NAME_TO_NEW[key]
    if new_idx is None:
        SKIP_CLASSES.add(old_idx)
    else:
        REMAP[old_idx] = new_idx

NEW_NAMES = {0: 'dent', 1: 'scratch', 2: 'glass', 3: 'broken_part'}
print('\nFinal remap:')
for old_idx in sorted(set(range(len(orig_names))) - SKIP_CLASSES):
    print(f"  {old_idx} ({orig_names[old_idx]:20s}) -> {REMAP[old_idx]} ({NEW_NAMES[REMAP[old_idx]]})")

In [ ]:
# Apply remapping + drop malformed annotations in one pass
DAMAGE_REMAPPED = Path('/kaggle/working/damage_remapped')
if DAMAGE_REMAPPED.exists():
    shutil.rmtree(DAMAGE_REMAPPED)
shutil.copytree(DAMAGE_RAW, DAMAGE_REMAPPED)

total_in = total_out = total_skipped = total_malformed = 0
for split in ['train', 'valid', 'test']:
    lbl_dir = DAMAGE_REMAPPED / split / 'labels'
    if not lbl_dir.exists():
        continue
    for txt in lbl_dir.glob('*.txt'):
        good_lines = []
        for line in txt.read_text().splitlines():
            if not line.strip():
                continue
            total_in += 1
            parts = line.split()
            old_cls = int(parts[0])
            if old_cls in SKIP_CLASSES:
                total_skipped += 1
                continue
            n_coords = len(parts) - 1
            if n_coords < 6 or n_coords % 2 != 0:
                total_malformed += 1
                continue
            new_cls = REMAP[old_cls]
            good_lines.append(' '.join([str(new_cls)] + parts[1:]))
            total_out += 1
        txt.write_text('\n'.join(good_lines) + ('\n' if good_lines else ''))

# Drop any stale label-cache files Ultralytics may have written
for cache_file in DAMAGE_REMAPPED.rglob('*.cache'):
    cache_file.unlink()

print(f"Annotations in:        {total_in}")
print(f"Annotations kept:      {total_out}")
print(f"Skipped (background):  {total_skipped}")
print(f"Dropped (malformed):   {total_malformed}")

In [ ]:
from collections import Counter

for split in ['train', 'valid', 'test']:
    lbl_dir = DAMAGE_REMAPPED / split / 'labels'
    if not lbl_dir.exists():
        continue
    counter = Counter()
    for txt in lbl_dir.glob('*.txt'):
        for line in txt.read_text().splitlines():
            if line.strip():
                counter[int(line.split()[0])] += 1
    total = sum(counter.values())
    print(f"\n{split} (total: {total} annotations):")
    for cls_idx in sorted(NEW_NAMES.keys()):
        n = counter.get(cls_idx, 0)
        pct = (n/total*100) if total > 0 else 0
        print(f"  {cls_idx} {NEW_NAMES[cls_idx]:12s}: {n:5d}  ({pct:5.1f}%)")

## 4. Write `data.yaml`

In [ ]:
data_yaml = {
    'path':  str(DAMAGE_REMAPPED),
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images' if (DAMAGE_REMAPPED / 'test' / 'images').exists() else 'valid/images',
    'names': NEW_NAMES,
}
yaml_path = Path('/kaggle/working/damage_data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, sort_keys=False)
print(yaml_path.read_text())

## 5. Train YOLOv8s-seg
60 epochs, AdamW, cos_lr, strong augmentation (mixup + copy_paste) to handle the wide visual variance of damage.

In [ ]:
model = YOLO('yolov8s-seg.pt')

results = model.train(
    data=str(yaml_path),
    epochs=60,
    imgsz=640,
    batch=16,
    patience=15,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    close_mosaic=10,
    mixup=0.2,
    copy_paste=0.15,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
    degrees=15.0,
    translate=0.15,
    scale=0.6,
    fliplr=0.5,
    cls=0.5,
    box=7.5,
    project='/kaggle/working/runs',
    name='damage_seg_v1',
    exist_ok=True,
    save=True,
    save_period=10,
    plots=True,
    verbose=True,
    device=0,
    workers=2,
    seed=42,
    amp=True,
)
print('\n✅ Training complete.')
print('Best weights:', '/kaggle/working/runs/damage_seg_v1/weights/best.pt')

## 6. Evaluate on the test split

In [ ]:
best_weights = '/kaggle/working/runs/damage_seg_v1/weights/best.pt'
best_model = YOLO(best_weights)

metrics = best_model.val(
    data=str(yaml_path),
    split='test' if (DAMAGE_REMAPPED / 'test' / 'images').exists() else 'val',
    imgsz=640,
    batch=16,
    plots=True,
    device=0,
)

import json
summary = {
    'box_mAP50':    float(metrics.box.map50),
    'box_mAP50-95': float(metrics.box.map),
    'mask_mAP50':   float(metrics.seg.map50),
    'mask_mAP50-95':float(metrics.seg.map),
    'mask_precision': float(metrics.seg.mp),
    'mask_recall':    float(metrics.seg.mr),
}
print(json.dumps(summary, indent=2))
Path('/kaggle/working/stage2_val_summary.json').write_text(json.dumps(summary, indent=2))

print('\nPer-class mask mAP50:')
for i, name in NEW_NAMES.items():
    print(f"  {name:12s}: {metrics.seg.maps[i]:.4f}")

## 7. Save the final model + create download link
After committing this notebook (Save Version → Save & Run All), the file shows up in the Output tab.

In [ ]:
from IPython.display import FileLink

src = Path(best_weights)
primary = Path('/kaggle/working/damage_seg_best.pt')
backup  = Path('/kaggle/working/STAGE2_BEST.pt')
shutil.copy(src, primary)
shutil.copy(src, backup)

print(f'Primary : {primary}  ({primary.stat().st_size/1e6:.1f} MB)')
print(f'Backup  : {backup}   ({backup.stat().st_size/1e6:.1f} MB)')

print('\nClick to download (fallback — use the Output tab if this link 404s):')
display(FileLink(str(primary)))

## 8. Qualitative inference on a few test images

In [ ]:
import matplotlib.pyplot as plt
import cv2

test_dir = DAMAGE_REMAPPED / ('test' if (DAMAGE_REMAPPED / 'test' / 'images').exists() else 'valid') / 'images'
test_imgs = list(test_dir.glob('*.jpg'))[:4]

preds = best_model.predict(test_imgs, imgsz=640, conf=0.25,
                            save=True, project='/kaggle/working/runs',
                            name='damage_inference', exist_ok=True, device=0)

fig, axes = plt.subplots(1, len(preds), figsize=(5*len(preds), 5))
if len(preds) == 1:
    axes = [axes]
for ax, r in zip(axes, preds):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis('off')
plt.tight_layout()
plt.show()